In [1]:
import pandas as pd
import pickle

# Load train_dataset 
train_data = pd.read_csv('datasets/train_dataset.csv')

# Separate features and target
X = train_data.drop(['isFraud'], axis=1)
y = train_data['isFraud']

# Load the 10-fold indices
with open("10_fold_indices.pkl", "rb") as f:
    splits = pickle.load(f)

print(f"Loaded {len(splits)} folds.")

Loaded 10 folds.


In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
)

# -------------------- Column Types --------------------
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# -------------------- Preprocessing Pipeline --------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# -------------------- Setup --------------------
os.makedirs("models/random_forest", exist_ok=True)

auc_scores = []
accuracy_scores = []
precision_0, precision_1 = [], []
recall_0, recall_1 = [], []
f1_0, f1_1 = [], []

# -------------------- Cross-Validation --------------------
for fold, (train_idx, val_idx) in enumerate(splits, 1):
    print(f"\n Fold {fold}")

    # Split data
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Preprocess
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)

    # Train model
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    rf.fit(X_train_processed, y_train)

    # Predict
    y_val_prob = rf.predict_proba(X_val_processed)[:, 1]
    y_val_pred = rf.predict(X_val_processed)

    # Evaluate
    auc = roc_auc_score(y_val, y_val_prob)
    acc = accuracy_score(y_val, y_val_pred)
    prec = precision_score(y_val, y_val_pred, average=None, zero_division=0)
    rec = recall_score(y_val, y_val_pred, average=None)
    f1 = f1_score(y_val, y_val_pred, average=None)

    auc_scores.append(auc)
    accuracy_scores.append(acc)
    precision_0.append(prec[0])
    precision_1.append(prec[1])
    recall_0.append(rec[0])
    recall_1.append(rec[1])
    f1_0.append(f1[0])
    f1_1.append(f1[1])

    # Print metrics
    print(f" Fold {fold} AUC-ROC : {auc:.4f}")
    print(f"   Accuracy     : {acc:.4f}")
    print(f"   Precision_0  : {prec[0]:.4f} | Precision_1 : {prec[1]:.4f}")
    print(f"   Recall_0     : {rec[0]:.4f} | Recall_1    : {rec[1]:.4f}")
    print(f"   F1 Score_0   : {f1[0]:.4f} | F1 Score_1  : {f1[1]:.4f}")

    # Save model
    with open(f"models/random_forest/fold_{fold}.pkl", "wb") as f:
        pickle.dump(rf, f)

# -------------------- Summary --------------------
print("\n Cross-Validation Summary (Random Forest + One-Hot Encoding + Scaling)")
for i in range(len(auc_scores)):
    print(
        f"Fold {i+1}: AUC={auc_scores[i]:.4f} | Acc={accuracy_scores[i]:.4f} | "
        f"Prec_0={precision_0[i]:.4f} | Prec_1={precision_1[i]:.4f} | "
        f"Recall_0={recall_0[i]:.4f} | Recall_1={recall_1[i]:.4f} | "
        f"F1_0={f1_0[i]:.4f} | F1_1={f1_1[i]:.4f}"
    )

print("\n Final Averages:")
print(f"AUC-ROC     : {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")
print(f"Accuracy    : {np.mean(accuracy_scores):.4f}")
print(f"Precision_0 : {np.mean(precision_0):.4f}")
print(f"Precision_1 : {np.mean(precision_1):.4f}")
print(f"Recall_0    : {np.mean(recall_0):.4f}")
print(f"Recall_1    : {np.mean(recall_1):.4f}")
print(f"F1 Score_0  : {np.mean(f1_0):.4f}")
print(f"F1 Score_1  : {np.mean(f1_1):.4f}")



 Fold 1
 Fold 1 AUC-ROC : 0.9332
   Accuracy     : 0.9783
   Precision_0  : 0.9787 | Precision_1 : 0.9505
   Recall_0     : 0.9992 | Recall_1    : 0.3998
   F1 Score_0   : 0.9889 | F1 Score_1  : 0.5629

 Fold 2
 Fold 2 AUC-ROC : 0.9329
   Accuracy     : 0.9784
   Precision_0  : 0.9790 | Precision_1 : 0.9439
   Recall_0     : 0.9991 | Recall_1    : 0.4076
   F1 Score_0   : 0.9889 | F1 Score_1  : 0.5693

 Fold 3
 Fold 3 AUC-ROC : 0.9447
   Accuracy     : 0.9794
   Precision_0  : 0.9798 | Precision_1 : 0.9550
   Recall_0     : 0.9993 | Recall_1    : 0.4313
   F1 Score_0   : 0.9894 | F1 Score_1  : 0.5942

 Fold 4
 Fold 4 AUC-ROC : 0.9284
   Accuracy     : 0.9782
   Precision_0  : 0.9784 | Precision_1 : 0.9642
   Recall_0     : 0.9995 | Recall_1    : 0.3906
   F1 Score_0   : 0.9888 | F1 Score_1  : 0.5560

 Fold 5
 Fold 5 AUC-ROC : 0.9315
   Accuracy     : 0.9787
   Precision_0  : 0.9791 | Precision_1 : 0.9582
   Recall_0     : 0.9994 | Recall_1    : 0.4105
   F1 Score_0   : 0.9891 | F1 Sco